# Headline-level -> Cross-Ticker Signal Assessment

This notebook walks through the complete pipeline for evaluating **raw news headline sentiment** as a predictor of asset price returns across multiple commodities and FX pairs.

## What This Notebook Does

1. **Loads** raw headline-level sentiment data from zip archives (one per ticker)
2. **Loads** price data and computes forward returns at multiple horizons (7d, 1m, 3m)
3. **Tests core filter variations** -- runs the full analysis pipeline for each headline quality filter (baseline, English+sentiment threshold, high topic confidence+low neutral), then picks the best-scoring filter per index
4. **Aggregates** individual headlines into daily sentiment indices per topic and per filter, deriving **4 sentiment metrics** from the raw scores
5. **Correlates** all 4 metrics x rolling transforms (z-score, mean) x 3 lookback windows with forward price returns across 3 regimes (ALL, UP market, DOWN market) -- separately for each filter variant
6. **Applies** multiple-testing corrections (Benjamini-Hochberg FDR) to identify statistically significant drivers
7. **Validates** signals with a 60/40 walk-forward in-sample / out-of-sample split
8. **Ranks** tickers, sentiment groups, and **metrics** in a cross-asset comparison table

## Sentiment Metrics

From each headline's `sentiment_score` we aggregate to daily level and derive **4 metrics**, each capturing a different aspect of sentiment:

| Metric | What it measures |
|---|---|
| `sentiment_avg` | Average sentiment direction (bullish/bearish) |
| `sentiment_std_daily` | Intra-day disagreement / dispersion across headlines |
| `headline_count` | Attention / news volume (number of headlines per day) |
| `conviction_ratio` | Net directional conviction -- ratio of net sentiment to total absolute sentiment (-1 to +1) |

**Key finding**: `conviction_ratio` and `sentiment_std_daily` frequently outperform the baseline `sentiment_avg`, producing stronger correlations and often better out-of-sample persistence. `headline_count` captures attention effects that pure sentiment direction misses.

## Input Data

**Headline Sentiment** (`asset_headlines/*.zip`): Each zip contains gzipped CSVs partitioned by `match_type` (EXPLICIT/IMPLICIT) and `year`. Each row is one headline with columns:

| Column | Description |
|---|---|
| `publication_time` | Timestamp of the headline |
| `language` | Language code (e.g. `en`, `de`) |
| `country` | Source country |
| `topic` | Sentiment topic (e.g. `Physical Demand-Transportation`) |
| `topic_probability` | Model confidence the headline belongs to this topic (0-1) |
| `sentiment_score` | Continuous score from -1 (bearish) to +1 (bullish) |
| `bearish_probability` | Model probability of bearish sentiment |
| `neutral_probability` | Model probability of neutral sentiment |
| `bullish_probability` | Model probability of bullish sentiment |

**Price Data** (`prices/prices.csv`): Daily prices with columns `ticker`, `trade_date`, `price`.

## How This Differs from the Indices Notebook

The **indices notebook** (`sentiment_indices_guide.ipynb`) works with **pre-aggregated hourly index data** where headlines have already been summarised into hourly counts and sentiment sums. This notebook starts from **raw headline-level data**, which enables:

- **Filter-variation testing** -- we can test whether filtering by `topic_probability`, `|sentiment_score|`, `neutral_probability`, or `language` improves signal quality before aggregation
- **Match-type partitioning** -- headlines are split into EXPLICIT (direct entity mention) and IMPLICIT (inferred relevance), plus a COMBINED view
- **Richer daily aggregation** -- we compute `sentiment_std_daily` directly from headline-level variance rather than averaging hourly std values

The correlation analysis, significance testing, walk-forward validation, and cross-ticker ranking steps are identical between the two notebooks.

## High-Level Output

- **Filter Effectiveness**: Which headline quality filter won each index, and how many indices each filter captures
- **Overall Ranking**: Each ticker graded A-F based on signal strength, correlation magnitude, and out-of-sample persistence
- **Metric Comparison**: Which of the 4 sentiment metrics produces the strongest / most persistent signals per ticker
- **Top Sentiment Drivers Per Ticker**: The single best-scoring sentiment topic for each asset, with the metric that produces it
- **Top Sentiment Group Drivers Per Ticker**: Aggregated by sentiment group (e.g. *Macro & Geopolitics*, *Physical Supply*) showing depth of signal
- **Regime Insights**: Which drivers are regime-specific vs regime-universal, and any sign-flipping patterns

## Step 0: Configuration & Imports

Set up paths, tickers to analyse, and the key parameters that control the correlation analysis.

In [ ]:
import warnings, zipfile, gzip, time
from pathlib import Path
from IPython.display import display as ipy_display

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as t_dist

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

BASE_DIR = Path(r"g:\My Drive\Code\Asset_Data_Analysis")
ASSET_HEADLINES_DIR = BASE_DIR / "asset_headlines"
PRICES_FILE = BASE_DIR / "prices" / "prices.csv"

TICKERS = ["QO_COM", "QC_COM", "ZS_COM", "AUD_IND", "BZ_COM", "NG_COM", "CL_COM"]

TICKER_LABELS = {
    "QO_COM": "Gold",
    "QC_COM": "Copper",
    "ZS_COM": "Soybeans",
    "AUD_IND": "Australian Dollar",
    "BZ_COM": "Brent Crude",
    "NG_COM": "Natural Gas",
    "CL_COM": "WTI Crude Oil",
}

RETURN_HORIZONS = {"7d": 7, "1m": 21, "3m": 63}
ROLLING_WINDOWS = [10, 21, 63]
SIGNIFICANCE_LEVEL = 0.05
REGIME_LOOKBACK = 63
REGIMES = ["BOTH", "UP", "DOWN"]
SPARSE_THRESHOLD_PCT = 15
WALK_FORWARD_IS_FRACTION = 0.60

SENTIMENT_METRICS = [
    "sentiment_avg",
    "sentiment_std_daily",
    "headline_count",
    "conviction_ratio",
]

CORE_FILTERS = [
    {"label": "baseline", "tp": 0.0, "abs_s": 0.0, "neut": 1.0, "lang": "all"},
    {"label": "|s|>=0.10+en", "tp": 0.0, "abs_s": 0.10, "neut": 1.0, "lang": "en"},
    {"label": "tp>=0.90+neut<=0.30", "tp": 0.90, "abs_s": 0.0, "neut": 0.30, "lang": "all"},
]

print(f"Tickers: {TICKERS}")
print(f"Sentiment metrics: {SENTIMENT_METRICS}")
print(f"Return horizons: {list(RETURN_HORIZONS.keys())}")
print(f"Rolling windows: {ROLLING_WINDOWS}")
print(f"Regimes: {REGIMES}  (lookback={REGIME_LOOKBACK}d)")
print(f"Walk-forward split: {WALK_FORWARD_IS_FRACTION * 100:.0f}% IS / {(1 - WALK_FORWARD_IS_FRACTION) * 100:.0f}% OOS")
print(f"Core filters: {len(CORE_FILTERS)} -- {[f['label'] for f in CORE_FILTERS]}")

## Step 1: Load Raw Headlines from Zip Files

Each ticker has a zip archive containing gzipped CSV files partitioned by `match_type` (EXPLICIT / IMPLICIT) and year. We read all partitions, tag each row with its `match_type`, and parse the timestamp.

In [ ]:
def find_zip_for_ticker(ticker: str) -> Path | None:
    for f in ASSET_HEADLINES_DIR.iterdir():
        if f.suffix == ".zip" and f.stem.endswith(f"_{ticker}"):
            return f
    return None


def load_headlines(zip_path: Path) -> pd.DataFrame:
    frames = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            parts = name.split("/")
            if len(parts) < 3:
                continue
            match_type = parts[0].replace("match_type=", "")
            with zf.open(name) as zipped_file:
                with gzip.open(zipped_file) as gz:
                    df = pd.read_csv(gz)
            df["match_type"] = match_type
            frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined["publication_time"] = pd.to_datetime(combined["publication_time"], format="ISO8601")
    return combined


all_headlines = {}
for ticker in TICKERS:
    zp = find_zip_for_ticker(ticker)
    if zp is None:
        print(f"WARNING: No zip found for {ticker}")
        continue
    df = load_headlines(zp)
    all_headlines[ticker] = df
    print(
        f"{ticker}: {len(df):>9,} headlines | topics: {df['topic'].nunique()} | "
        f"date range: {df['publication_time'].min().date()} to {df['publication_time'].max().date()}"
    )

sample_ticker = TICKERS[0]
print(f"\nSample headlines ({sample_ticker}):")
all_headlines[sample_ticker][["publication_time", "topic", "sentiment_score", "match_type"]].head(8)

## Step 2: Load & Prepare Price Data

We load daily closing prices from `prices/prices.csv` (columns: `ticker`, `trade_date`, `price`), match each ticker, compute **forward returns** at multiple horizons (7d, 1m, 3m), and classify each day into an **UP or DOWN regime** based on the trailing 63-day return.

In [ ]:
def load_and_prepare_prices(ticker: str, prices: pd.DataFrame) -> pd.DataFrame | None:
    candidates = [ticker, ticker.replace("_", " "), ticker.split("_")[0] + " " + "_".join(ticker.split("_")[1:])]
    prices_ticker = None
    for c in candidates:
        subset = prices[prices["ticker"] == c]
        if not subset.empty:
            prices_ticker = subset.copy()
            break
    if prices_ticker is None:
        partial = [t for t in prices["ticker"].unique() if ticker.split("_")[0] in t]
        if partial:
            prices_ticker = prices[prices["ticker"] == partial[0]].copy()
    if prices_ticker is None:
        return None

    prices_ticker["trade_date"] = pd.to_datetime(prices_ticker["trade_date"])
    prices_ticker = prices_ticker.sort_values("trade_date").set_index("trade_date")

    price_col = "ask_close" if "ask_close" in prices_ticker.columns else "price"

    for label, days in RETURN_HORIZONS.items():
        prices_ticker[f"fwd_ret_{label}"] = prices_ticker[price_col].shift(-days) / prices_ticker[price_col] - 1

    trailing = prices_ticker[price_col] / prices_ticker[price_col].shift(REGIME_LOOKBACK) - 1
    prices_ticker["regime"] = np.where(trailing >= 0, "UP", "DOWN")
    prices_ticker.loc[trailing.isna(), "regime"] = np.nan
    return prices_ticker


raw_prices = pd.read_csv(PRICES_FILE)
all_prices = {}
for ticker in TICKERS:
    pt = load_and_prepare_prices(ticker, raw_prices)
    if pt is None:
        print(f"WARNING: No price data for {ticker}")
        continue
    all_prices[ticker] = pt
    rc = pt["regime"].value_counts()
    print(f"{ticker}: {len(pt)} price days | UP={rc.get('UP', 0)} DOWN={rc.get('DOWN', 0)}")

print(f"\nSample prices ({sample_ticker}):")
_pc = "ask_close" if "ask_close" in all_prices[sample_ticker].columns else "price"
all_prices[sample_ticker][[_pc, "fwd_ret_7d", "fwd_ret_1m", "regime"]].head(5)

## Step 3: Headline Quality & Filter Variations

Before running the full correlation analysis, we check whether **filtering headlines** improves signal quality. The raw data contains headlines of varying quality:

- **`topic_probability`** — how confident the model is that the headline belongs to the assigned topic. Low values mean the topic assignment may be wrong.
- **`|sentiment_score|`** — the absolute strength of sentiment. Scores near zero are essentially neutral and may add noise without information.
- **`neutral_probability`** — the model’s estimate that the headline carries no sentiment. High values suggest the headline is uninformative.
- **`language`** — non-English headlines may have lower sentiment model accuracy.

We test several filter combinations and compare each against the **baseline** (no filtering). For each variation, we run the full aggregation + correlation pipeline and count how many BH-significant drivers emerge. The key metric is **BH excess** — the number of significant results above what we’d expect by chance (5% false positive rate).

| Variation | What it does |
|---|---|
| `baseline` | No filtering — all headlines included |
| `tp>=0.90` | Keep only headlines with high topic confidence |
| `\|s\|>=0.10+en` | Remove near-neutral headlines, English only |
| `tp>=0.90+neut<=0.30` | High topic confidence + remove likely-neutral |
| `tp>=0.95+\|s\|>=0.30+neut<=0.30+en` | Aggressive: high confidence, strong sentiment, English only |

If a stricter filter produces **more** excess significant results with **fewer** headlines, it means we’re removing noise and concentrating signal. If it produces fewer, the removed headlines were actually contributing useful information.

In [ ]:
def fast_aggregate_to_daily(headlines: pd.DataFrame) -> pd.DataFrame:
    """Aggregate headlines straight to daily sentiment with all 4 metrics."""
    headlines = headlines.copy()
    headlines["date"] = headlines["publication_time"].dt.normalize()

    agg = (
        headlines.groupby(["date", "match_type", "topic"])["sentiment_score"]
        .agg(["sum", "count", "std", lambda x: x.abs().sum()])
        .reset_index()
    )
    agg.columns = [
        "date",
        "index_type",
        "topic",
        "sentiment_sum",
        "headline_count",
        "sentiment_std_daily",
        "sentiment_abs_sum",
    ]
    agg["sentiment_std_daily"] = agg["sentiment_std_daily"].fillna(0)
    agg["sentiment_avg"] = agg["sentiment_sum"] / agg["headline_count"]
    agg["conviction_ratio"] = agg["sentiment_sum"] / agg["sentiment_abs_sum"].replace(0, np.nan)

    comb = (
        headlines.groupby(["date", "topic"])["sentiment_score"]
        .agg(["sum", "count", "std", lambda x: x.abs().sum()])
        .reset_index()
    )
    comb.columns = ["date", "topic", "sentiment_sum", "headline_count", "sentiment_std_daily", "sentiment_abs_sum"]
    comb["sentiment_std_daily"] = comb["sentiment_std_daily"].fillna(0)
    comb["sentiment_avg"] = comb["sentiment_sum"] / comb["headline_count"]
    comb["conviction_ratio"] = comb["sentiment_sum"] / comb["sentiment_abs_sum"].replace(0, np.nan)
    comb["index_type"] = "COMBINED"

    daily = pd.concat([agg, comb], ignore_index=True)
    daily["group"] = daily["topic"].str.split("-").str[0]

    group_agg = (
        daily.groupby(["date", "index_type", "group"])
        .agg(
            sentiment_sum=("sentiment_sum", "sum"),
            headline_count=("headline_count", "sum"),
            sentiment_abs_sum=("sentiment_abs_sum", "sum"),
            sentiment_std_daily=("sentiment_std_daily", "mean"),
        )
        .reset_index()
    )
    group_agg["sentiment_avg"] = group_agg["sentiment_sum"] / group_agg["headline_count"]
    group_agg["conviction_ratio"] = group_agg["sentiment_sum"] / group_agg["sentiment_abs_sum"].replace(0, np.nan)
    group_agg.rename(columns={"group": "name"}, inplace=True)
    group_agg["topic_level"] = "GROUP"

    metric_cols = ["date", "index_type", "name", "topic_level"] + SENTIMENT_METRICS
    topic_part = daily[["date", "index_type", "topic"] + [c for c in SENTIMENT_METRICS if c in daily.columns]].copy()
    topic_part.rename(columns={"topic": "name"}, inplace=True)
    topic_part["topic_level"] = "TOPIC"

    return pd.concat(
        [
            group_agg[[c for c in metric_cols if c in group_agg.columns]],
            topic_part[[c for c in metric_cols if c in topic_part.columns]],
        ],
        ignore_index=True,
    )


def apply_transforms(series: pd.Series, window: int) -> dict[str, pd.Series]:
    roll_mean = series.rolling(window, min_periods=max(1, window // 2)).mean()
    roll_std = series.rolling(window, min_periods=max(1, window // 2)).std()
    return {
        "rolling_zscore": (series - roll_mean) / roll_std.replace(0, np.nan),
        "rolling_mean": roll_mean,
    }


def vectorized_correlation_sweep(sentiment_all: pd.DataFrame, prices_ticker: pd.DataFrame) -> pd.DataFrame:
    fwd_ret_cols = [f"fwd_ret_{h}" for h in RETURN_HORIZONS]
    horizon_names = list(RETURN_HORIZONS.keys())
    prices_indexed = prices_ticker[["regime"] + fwd_ret_cols].copy()
    regime_col = prices_indexed["regime"]
    ret_matrix = prices_indexed[fwd_ret_cols].values
    price_index = prices_indexed.index

    results = []
    for (topic_level, index_type, name), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        grp = grp.set_index("date").sort_index()
        full_idx = pd.date_range(grp.index.min(), grp.index.max(), freq="B")

        for metric in SENTIMENT_METRICS:
            if metric not in grp.columns:
                continue
            fill_val = 0 if metric != "conviction_ratio" else np.nan
            raw_series = grp[metric].reindex(full_idx).fillna(fill_val)
            if raw_series.dropna().nunique() < 3:
                continue

            for window in ROLLING_WINDOWS:
                transformed = apply_transforms(raw_series, window)
                for transform_name, t_series in transformed.items():
                    common_idx = t_series.dropna().index.intersection(price_index)
                    if len(common_idx) < 30:
                        continue
                    s_vals = t_series.loc[common_idx].values
                    p_loc = price_index.get_indexer(common_idx)
                    r_vals = ret_matrix[p_loc]
                    reg_vals = regime_col.values[p_loc]

                    for regime in REGIMES:
                        rmask = np.ones(len(common_idx), dtype=bool) if regime == "BOTH" else (reg_vals == regime)
                        s_reg, r_reg = s_vals[rmask], r_vals[rmask]
                        if rmask.sum() < 30:
                            continue
                        s_finite = np.isfinite(s_reg)
                        for h_idx, horizon in enumerate(horizon_names):
                            r_col = r_reg[:, h_idx]
                            valid = s_finite & np.isfinite(r_col)
                            n_h = valid.sum()
                            if n_h < 30:
                                continue
                            sx, ry = s_reg[valid], r_col[valid]
                            sx_m, ry_m = sx - sx.mean(), ry - ry.mean()
                            ss_x, ss_y = (sx_m**2).sum(), (ry_m**2).sum()
                            if ss_x == 0 or ss_y == 0:
                                continue
                            corr = np.clip((sx_m * ry_m).sum() / np.sqrt(ss_x * ss_y), -1, 1)
                            if abs(corr) < 1:
                                t_stat = corr * np.sqrt((n_h - 2) / (1 - corr**2))
                                pval = 2 * t_dist.sf(abs(t_stat), n_h - 2)
                            else:
                                t_stat, pval = np.inf, 0.0
                            results.append(
                                (
                                    regime,
                                    metric,
                                    topic_level,
                                    index_type,
                                    name,
                                    transform_name,
                                    window,
                                    horizon,
                                    int(n_h),
                                    round(corr, 6),
                                    pval,
                                    round(t_stat, 4),
                                )
                            )

    if not results:
        return pd.DataFrame()
    return pd.DataFrame(
        results,
        columns=[
            "regime",
            "metric",
            "topic_level",
            "index_type",
            "name",
            "transform",
            "window",
            "return_horizon",
            "n_obs",
            "pearson_r",
            "p_value",
            "t_stat",
        ],
    )


def benjamini_hochberg(p_values: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    n = len(p_values)
    if n == 0:
        return np.array([], dtype=bool)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    thresholds = alpha * np.arange(1, n + 1) / n
    below = sorted_p <= thresholds
    if not below.any():
        return np.zeros(n, dtype=bool)
    max_idx = np.max(np.where(below))
    significant = np.zeros(n, dtype=bool)
    significant[sorted_idx[: max_idx + 1]] = True
    return significant


def apply_significance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    significance = pd.Series("not significant", index=df.index)
    significance[df["p_value"] < SIGNIFICANCE_LEVEL] = "significant"
    for regime in REGIMES:
        regime_idx = df.index[df["regime"] == regime]
        if len(regime_idx) == 0:
            continue
        bh_mask = benjamini_hochberg(df.loc[regime_idx, "p_value"].values, SIGNIFICANCE_LEVEL)
        significance.loc[regime_idx[bh_mask]] = "significant (BH adjusted)"
    df["significance"] = significance
    return df


def quick_significance_counts(sentiment_all: pd.DataFrame, prices_ticker: pd.DataFrame) -> dict:
    empty = {
        "total_tests": 0,
        "n_significant_bh": 0,
        "expected_fp": 0,
        "best_abs_r": np.nan,
        "median_abs_r_sig": np.nan,
        "n_sig_indices": 0,
    }
    results_df = vectorized_correlation_sweep(sentiment_all, prices_ticker)
    if results_df.empty:
        return empty
    results_df = apply_significance(results_df)
    total = len(results_df)
    bh = results_df[results_df["significance"] == "significant (BH adjusted)"]
    n_bh = len(bh)
    n_sig_indices = bh.groupby(["topic_level", "index_type", "name"]).ngroups if n_bh else 0
    return {
        "total_tests": total,
        "n_significant_bh": int(n_bh),
        "expected_fp": round(total * SIGNIFICANCE_LEVEL, 1),
        "best_abs_r": round(float(bh["pearson_r"].abs().max()), 4) if n_bh else np.nan,
        "median_abs_r_sig": round(float(bh["pearson_r"].abs().median()), 4) if n_bh else np.nan,
        "n_sig_indices": n_sig_indices,
    }


def apply_headline_filter(headlines: pd.DataFrame, filt: dict) -> pd.DataFrame:
    """Apply a CORE_FILTERS entry to raw headlines and return the filtered subset."""
    mask = pd.Series(True, index=headlines.index)
    if filt["tp"] > 0:
        mask &= headlines["topic_probability"] >= filt["tp"]
    if filt["abs_s"] > 0:
        mask &= headlines["sentiment_score"].abs() >= filt["abs_s"]
    if filt["neut"] < 1.0:
        mask &= headlines["neutral_probability"] <= filt["neut"]
    if filt["lang"] != "all":
        mask &= headlines["language"] == filt["lang"]
    return headlines.loc[mask]


def run_filter_variations(headlines: pd.DataFrame, prices_ticker: pd.DataFrame) -> pd.DataFrame:
    n_total = len(headlines)
    is_english = headlines["language"].values == "en"
    abs_sent = headlines["sentiment_score"].abs().values
    tp_vals = headlines["topic_probability"].values
    neut_vals = headlines["neutral_probability"].values

    rows = []
    for var in CORE_FILTERS:
        t0 = time.perf_counter()
        mask = np.ones(n_total, dtype=bool)
        if var["tp"] > 0:
            mask &= tp_vals >= var["tp"]
        if var["abs_s"] > 0:
            mask &= abs_sent >= var["abs_s"]
        if var["neut"] < 1.0:
            mask &= neut_vals <= var["neut"]
        if var["lang"] != "all":
            mask &= is_english

        n_kept = int(mask.sum())
        if n_kept < 50:
            rows.append(
                {
                    "variation": var["label"],
                    "headlines_kept": n_kept,
                    "pct_kept": round(100 * n_kept / max(n_total, 1), 1),
                    "total_tests": 0,
                    "n_significant_bh": 0,
                    "expected_fp": 0,
                    "bh_excess": 0,
                    "sig_rate_pct": 0.0,
                    "best_abs_r": np.nan,
                    "median_abs_r_sig": np.nan,
                    "n_sig_indices": 0,
                }
            )
            continue

        filtered = headlines.loc[mask]
        sentiment_all = fast_aggregate_to_daily(filtered)
        counts = quick_significance_counts(sentiment_all, prices_ticker)

        bh_excess = counts["n_significant_bh"] - counts["expected_fp"]
        sig_rate = 100 * counts["n_significant_bh"] / max(counts["total_tests"], 1)
        elapsed = time.perf_counter() - t0

        rows.append(
            {
                "variation": var["label"],
                "headlines_kept": n_kept,
                "pct_kept": round(100 * n_kept / max(n_total, 1), 1),
                **counts,
                "bh_excess": round(bh_excess, 1),
                "sig_rate_pct": round(sig_rate, 2),
            }
        )
        print(
            f"    {var['label']:45s} => {n_kept:>9,} ({100 * n_kept / n_total:>5.1f}%) "
            f"BH={counts['n_significant_bh']:>4} "
            f"excess={bh_excess:>+5.0f} "
            f"rate={sig_rate:>5.1f}%  [{elapsed:.1f}s]"
        )

    return pd.DataFrame(rows)


# Run filter variations for each ticker
all_filter_results = {}
for ticker in TICKERS:
    if ticker not in all_headlines or ticker not in all_prices:
        continue
    print(f"\n--- {ticker} ---")
    var_df = run_filter_variations(all_headlines[ticker], all_prices[ticker])
    all_filter_results[ticker] = var_df

# Cross-ticker summary: did any filter beat baseline?
rename_cols = {
    "variation": "Filter",
    "headlines_kept": "Headlines",
    "pct_kept": "% Kept",
    "n_significant_bh": "BH Sig",
    "expected_fp": "Expected FP",
    "bh_excess": "Excess",
    "sig_rate_pct": "Sig Rate %",
    "best_abs_r": "Best |r|",
}
display_fv_cols = [
    "variation",
    "headlines_kept",
    "pct_kept",
    "n_significant_bh",
    "expected_fp",
    "bh_excess",
    "sig_rate_pct",
    "best_abs_r",
]

improved_tickers = []
print("\nCross-ticker filter summary:\n")
for ticker in TICKERS:
    if ticker not in all_filter_results:
        continue
    vdf = all_filter_results[ticker]
    baseline = vdf[vdf["variation"] == "baseline"]
    b_excess = baseline.iloc[0]["bh_excess"] if not baseline.empty else 0
    best = vdf.loc[vdf["bh_excess"].idxmax()]
    if best["bh_excess"] > b_excess * 1.1:
        improved_tickers.append(ticker)
        print(
            f"  {ticker}: BEST = '{best['variation']}' (excess={best['bh_excess']:+.0f}, "
            f"vs baseline {b_excess:+.0f}, using {best['pct_kept']:.0f}% of headlines)"
        )
    else:
        print(f"  {ticker}: Baseline is best or near-best (excess={b_excess:+.0f})")

# Show full filter tables for tickers where filtering helps
if improved_tickers:
    print(f"\nDetailed results for tickers where filtering improves signal:\n")
    for ticker in improved_tickers:
        print(f"  {ticker} ({TICKER_LABELS.get(ticker, ticker)}):")
        print(all_filter_results[ticker][display_fv_cols].rename(columns=rename_cols).to_string(index=False))
        print()
else:
    print("\nBaseline (no filtering) is optimal for all tickers.")

## Step 4: Aggregate Headlines to Daily Sentiment Indices (Per Filter)

Rather than using only the baseline (unfiltered) headlines, we now run the full aggregation for **each core filter**. For each filter variant, we aggregate headlines into daily sentiment indices, then in later steps run correlations and walk-forward validation separately. The final index summary picks the **best-scoring filter** per index -- meaning different sentiment topics may benefit from different quality thresholds.

For each `(date, index_type, topic)` we aggregate headline-level scores into four daily sentiment metrics:

| Metric | Definition |
|---|---|
| `sentiment_avg` | Mean `sentiment_score` across all headlines for the day — captures the directional consensus (positive = bullish, negative = bearish) |
| `sentiment_std_daily` | Standard deviation of `sentiment_score` within the day — measures disagreement among headlines (high = mixed views, low = consensus) |
| `headline_count` | Number of headlines published that day — proxies for attention / news intensity on the topic |
| `conviction_ratio` | Sum of raw scores ÷ sum of absolute scores — ranges from −1 to +1 and measures how one-sided sentiment is after accounting for volume (values near ±1 mean almost all headlines agree in direction) |

### Index types: EXPLICIT, IMPLICIT, COMBINED

Each headline is tagged with a `match_type` that describes how it was linked to the asset:

- **EXPLICIT** — the headline directly names the asset (e.g. *"Gold prices surge on safe-haven demand"*). These have the clearest relevance but may be priced in faster by markets.
- **IMPLICIT** — the headline is topically related but does not name the asset directly (e.g. *"Fed signals pause in rate hikes"* linked to gold via the Central Banking topic). These capture broader macro and thematic drivers that may contain slower-moving, less crowded signal.
- **COMBINED** — the union of EXPLICIT and IMPLICIT headlines, aggregated together. This maximises headline volume and provides the most robust daily estimates, at the cost of mixing direct and indirect references. For tickers that only have IMPLICIT headlines (e.g. AUD_IND), COMBINED is identical to IMPLICIT.

All three index types are carried through the analysis in parallel so we can compare whether direct mentions, indirect context, or the blend produces the strongest predictive signal.

### Topic levels

We split into two granularities:

- **GROUP**: the prefix before the first `-` (e.g. `Physical Supply`)
- **TOPIC**: the full topic name (e.g. `Physical Supply-Mine Production`)

This produces a unified `sentiment_all` table that feeds every downstream analysis.

### Coverage & sparse indices

Not every topic has headlines every day. We compute **coverage** — the percentage of business days that have at least one headline for a given `(index_type, topic)`. Indices with coverage below **15%** are flagged as **sparse**.

Sparse indices are still included in the analysis (excluding them would bias results toward high-volume topics), but they should be interpreted with caution:

- Correlations are computed on fewer data points, so they have wider confidence intervals
- A high |r| on a sparse index may reflect a short, unusual period rather than a persistent relationship
- In the output tables, sparse indices are flagged so you can distinguish between a strong signal on dense data (high confidence) vs a strong signal on thin data (needs more validation)

In [ ]:
all_sentiment = {}  # {(ticker, filter_label): sentiment_df}
for ticker in TICKERS:
    if ticker not in all_headlines:
        continue
    for filt in CORE_FILTERS:
        label = filt["label"]
        if label == "baseline":
            filtered = all_headlines[ticker]
        else:
            filtered = apply_headline_filter(all_headlines[ticker], filt)
        n_kept = len(filtered)
        pct = 100 * n_kept / max(len(all_headlines[ticker]), 1)
        if n_kept < 50:
            print(f"{ticker} / {label}: SKIPPED ({n_kept} headlines)")
            continue
        sent = fast_aggregate_to_daily(filtered)
        all_sentiment[(ticker, label)] = sent
        n_idx = sent.groupby(["topic_level", "index_type", "name"]).ngroups
        print(f"{ticker} / {label}: {n_idx} indices, {len(sent):,} daily rows ({pct:.0f}% of headlines)")

print(f"\nTotal (ticker, filter) combinations: {len(all_sentiment)}")
print(f"\nSample daily sentiment ({sample_ticker}, baseline):")
all_sentiment[(sample_ticker, "baseline")].head(8)

## Step 5: Compute Coverage & Run Correlation Analysis

For every combination of `(regime, topic_level, index_type, name, transform, window, return_horizon)` we:

1. Apply a **rolling transform** to the daily sentiment series
2. Align with forward price returns
3. Compute **Pearson correlation** and a two-tailed **t-test** p-value

### Why Pearson correlation?

We use **Pearson's r** to measure the linear relationship between a transformed sentiment series and forward asset returns. It answers: *"when sentiment is higher than usual, do prices tend to move in a consistent direction?"*

- **r > 0**: positive sentiment predicts positive returns (bullish signal)
- **r < 0**: positive sentiment predicts negative returns (contrarian signal)
- **|r|**: the strength of the relationship — values above 0.10 are noteworthy in financial data, above 0.30 is strong

The **p-value** comes from a t-test on the correlation coefficient: `t = r × √((n-2) / (1-r²))` with `n-2` degrees of freedom. This tests whether the observed correlation is statistically distinguishable from zero given the sample size. With ~2,000 trading days, even small correlations (|r| ≈ 0.05) can be statistically significant — which is why we also need the multiple-testing correction in the next step.

### Why these two transforms?

- **Rolling mean** — smooths out daily noise to reveal the underlying sentiment trend. A 21-day rolling mean, for example, captures whether sentiment has been persistently positive or negative over the past month, which is more likely to predict medium-term returns than a single day's reading.
- **Rolling z-score** — normalises sentiment relative to its own recent history (`(value − rolling_mean) / rolling_std`). This detects *unusual* sentiment shifts regardless of the topic's baseline level. A z-score of +2 means sentiment is unusually bullish compared to its recent past, which may signal a regime change that prices haven't yet fully reflected.

We sweep across multiple rolling windows (5, 10, 21, 63 days) to capture signals at different frequencies — from short-term momentum (weekly) to longer structural shifts (quarterly).

This is the most compute-intensive step — we use a vectorised implementation for speed.

In [ ]:
def compute_coverage(sentiment_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (tl, it, nm), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        bdays = len(pd.bdate_range(grp["date"].min(), grp["date"].max()))
        n_days = grp["date"].nunique()
        rows.append(
            {
                "topic_level": tl,
                "index_type": it,
                "name": nm,
                "days_with_data": n_days,
                "total_bdays": bdays,
                "coverage_pct": round(100 * n_days / max(bdays, 1), 1),
            }
        )
    return pd.DataFrame(rows)


all_corr_results = {}  # {(ticker, filter_label): results_df}
all_coverage = {}  # {(ticker, filter_label): coverage_df}
for (ticker, label), sent in all_sentiment.items():
    if ticker not in all_prices:
        continue
    print(f"\n--- {ticker} / {label} ---")
    cov = compute_coverage(sent)
    all_coverage[(ticker, label)] = cov
    res = vectorized_correlation_sweep(sent, all_prices[ticker])
    res["ticker"] = ticker
    res["filter_label"] = label
    cov_map = cov.set_index(["topic_level", "index_type", "name"])["coverage_pct"]
    res["coverage_pct"] = res.set_index(["topic_level", "index_type", "name"]).index.map(cov_map).values
    res["coverage_pct"] = res["coverage_pct"].fillna(0)
    all_corr_results[(ticker, label)] = res
    print(f"  {len(res):,} correlation tests")

print(f"\nTotal (ticker, filter) test sets: {len(all_corr_results)}")
print(f"Total tests: {sum(len(v) for v in all_corr_results.values()):,}")

## Step 6: Apply Significance Testing (Benjamini-Hochberg FDR)

### The multiple-testing problem

With ~17,000 correlation tests per ticker (56–87 sentiment indices × 4 metrics × 2 transforms × 3 windows × 3 horizons × 3 regimes), we expect **~850 false positives** at p < 0.05 even if sentiment has *zero* predictive power. Simply counting results with p < 0.05 is meaningless without correction.

The index count varies by ticker because it is the number of unique `(topic_level, index_type, name)` combinations — driven by how many topics exist for that asset and whether EXPLICIT headlines are available (e.g. AUD_IND has no EXPLICIT headlines, so COMBINED is identical to IMPLICIT and the effective index count drops to 56 vs ~84 for commodities with all three distinct index types).

### How we address it

1. **Raw p-value** threshold at 0.05 — a first pass to flag candidates
2. **Benjamini-Hochberg (BH) FDR correction** — applied per regime, this procedure controls the **false discovery rate** rather than the family-wise error rate. It ranks all p-values, then finds the largest rank *k* where `p(k) ≤ 0.05 × k / N`. All tests with rank ≤ k are declared significant. This is less conservative than Bonferroni (which would require p < 0.000003) but still rigorous.

### How to read the output

- **BH sig**: number of tests surviving the BH correction
- **Expected FP**: 5% × total tests — what we'd expect by chance alone
- **Excess**: BH sig − Expected FP — the number of significant results *above* what chance predicts. A large positive excess is strong evidence of genuine signal.

Only BH-adjusted significant results are used for ranking.

In [ ]:
for key in list(all_corr_results.keys()):
    ticker, label = key
    all_corr_results[key] = apply_significance(all_corr_results[key])
    counts = all_corr_results[key]["significance"].value_counts()
    total = len(all_corr_results[key])
    n_bh = counts.get("significant (BH adjusted)", 0)
    expected_fp = total * SIGNIFICANCE_LEVEL
    print(
        f"{ticker}/{label}: {total:,} tests | BH sig: {n_bh:,} ({100 * n_bh / total:.1f}%) | "
        f"Expected FP: {expected_fp:.0f} | Excess: {n_bh - expected_fp:+.0f}"
    )

## Step 7: Build Index Summary & Walk-Forward Validation

For each `(topic_level, index_type, name)` we build a summary row with:
- Coverage stats and a sparse flag
- Per-regime best correlation, p-value, and significant horizons
- A **composite score** = `Σ -log10(p) × |r|` across regimes

Then we run a **60/40 walk-forward split**: find significant drivers in the in-sample period, and check how many retain BH-significance out-of-sample.

In [ ]:
def build_index_summary(results_df: pd.DataFrame, coverage_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    sig = results_df[results_df["significance"] == "significant (BH adjusted)"].copy()
    if not sig.empty:
        sig["abs_r"] = sig["pearson_r"].abs()

    sig_grouped = {}
    if not sig.empty:
        for (tl, it, nm, regime), grp in sig.groupby(["topic_level", "index_type", "name", "regime"]):
            sig_grouped[(tl, it, nm, regime)] = grp

    horizon_order = list(RETURN_HORIZONS.keys())
    rows = []
    for _, cov_row in coverage_df.iterrows():
        tl, it, nm = cov_row["topic_level"], cov_row["index_type"], cov_row["name"]
        row = {
            "ticker": ticker,
            "topic_level": tl,
            "name": nm,
            "index_type": it,
            "days_with_data": cov_row["days_with_data"],
            "total_bdays": cov_row["total_bdays"],
            "coverage_pct": cov_row["coverage_pct"],
            "sparse": cov_row["coverage_pct"] < SPARSE_THRESHOLD_PCT,
        }
        score = 0.0
        for regime in REGIMES:
            r_sig = sig_grouped.get((tl, it, nm, regime))
            if r_sig is None or r_sig.empty:
                row[f"{regime}_sig_horizons"] = ""
                row[f"{regime}_best_r"] = np.nan
                row[f"{regime}_best_t"] = np.nan
                row[f"{regime}_best_p"] = np.nan
                row[f"{regime}_best_metric"] = ""
            else:
                horizons = sorted(r_sig["return_horizon"].unique(), key=lambda h: horizon_order.index(h))
                row[f"{regime}_sig_horizons"] = "|".join(horizons)
                best_idx = r_sig["abs_r"].idxmax()
                best_r = r_sig.loc[best_idx, "pearson_r"]
                best_p = r_sig.loc[best_idx, "p_value"]
                row[f"{regime}_best_r"] = round(best_r, 4)
                row[f"{regime}_best_t"] = r_sig.loc[best_idx, "t_stat"]
                row[f"{regime}_best_p"] = best_p
                row[f"{regime}_best_metric"] = r_sig.loc[best_idx, "metric"]
                score += -np.log10(max(best_p, 1e-300)) * abs(best_r)

        sig_regimes = [r for r in REGIMES if row.get(f"{r}_sig_horizons", "") != ""]
        row["sig_regimes"] = "|".join(sig_regimes) if sig_regimes else ""

        idx_sig = (
            sig[(sig["topic_level"] == tl) & (sig["index_type"] == it) & (sig["name"] == nm)]
            if not sig.empty
            else pd.DataFrame()
        )
        best_metric = ""
        best_abs_r = 0
        for metric in SENTIMENT_METRICS:
            m_sig = idx_sig[idx_sig["metric"] == metric] if not idx_sig.empty else pd.DataFrame()
            row[f"{metric}_n_sig"] = len(m_sig)
            if not m_sig.empty:
                m_best_idx = m_sig["abs_r"].idxmax()
                row[f"{metric}_best_r"] = round(m_sig.loc[m_best_idx, "pearson_r"], 4)
                if m_sig.loc[m_best_idx, "abs_r"] > best_abs_r:
                    best_abs_r = m_sig.loc[m_best_idx, "abs_r"]
                    best_metric = metric
            else:
                row[f"{metric}_best_r"] = np.nan
        row["best_metric"] = best_metric

        row["score"] = round(score, 2)
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.sort_values("score", ascending=False, inplace=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    return summary


def walk_forward_validate(sentiment_all: pd.DataFrame, prices_ticker: pd.DataFrame, ticker: str) -> pd.DataFrame:
    all_dates = prices_ticker.index.sort_values()
    n_total = len(all_dates)
    n_is = int(n_total * WALK_FORWARD_IS_FRACTION)
    split_date = all_dates[n_is - 1]
    is_start, is_end = all_dates[0], split_date
    oos_start, oos_end = all_dates[n_is], all_dates[-1]

    is_sent = sentiment_all[(sentiment_all["date"] >= is_start) & (sentiment_all["date"] <= is_end)]
    is_prices = prices_ticker.loc[is_start:is_end]
    is_results = apply_significance(vectorized_correlation_sweep(is_sent, is_prices))
    if is_results.empty:
        return pd.DataFrame()

    oos_sent = sentiment_all[(sentiment_all["date"] >= oos_start) & (sentiment_all["date"] <= oos_end)]
    oos_prices = prices_ticker.loc[oos_start:oos_end]
    oos_results = apply_significance(vectorized_correlation_sweep(oos_sent, oos_prices))
    if oos_results.empty:
        return pd.DataFrame()

    def _key(row):
        return (
            row["regime"],
            row["metric"],
            row["topic_level"],
            row["index_type"],
            row["name"],
            row["transform"],
            row["window"],
            row["return_horizon"],
        )

    is_sig = is_results[is_results["significance"] == "significant (BH adjusted)"].copy()
    is_sig["key"] = is_sig.apply(_key, axis=1)
    oos_results["key"] = oos_results.apply(_key, axis=1)
    oos_lookup = {row["key"]: row for _, row in oos_results.iterrows()}

    matched = []
    for _, is_row in is_sig.iterrows():
        k = is_row["key"]
        if k in oos_lookup:
            oos_row = oos_lookup[k]
            matched.append(
                {
                    "regime": is_row["regime"],
                    "metric": is_row["metric"],
                    "topic_level": is_row["topic_level"],
                    "index_type": is_row["index_type"],
                    "name": is_row["name"],
                    "transform": is_row["transform"],
                    "window": is_row["window"],
                    "return_horizon": is_row["return_horizon"],
                    "is_r": is_row["pearson_r"],
                    "is_p": is_row["p_value"],
                    "oos_r": oos_row["pearson_r"],
                    "oos_p": oos_row["p_value"],
                    "oos_sig": oos_row["significance"],
                }
            )
    return pd.DataFrame(matched)


def enrich_with_walkforward(idx_summary: pd.DataFrame, wf_df: pd.DataFrame) -> pd.DataFrame:
    idx_summary = idx_summary.copy()
    for col, val in {
        "wf_is_sig": 0,
        "wf_oos_persistent": 0,
        "wf_oos_pct": 0.0,
        "wf_same_sign": 0,
        "wf_sign_rate": 0.0,
        "wf_best_oos_r": np.nan,
        "wf_best_oos_p": np.nan,
    }.items():
        idx_summary[col] = val
    if wf_df.empty:
        return idx_summary
    for (tl, it, nm), grp in wf_df.groupby(["topic_level", "index_type", "name"]):
        mask = (idx_summary["topic_level"] == tl) & (idx_summary["index_type"] == it) & (idx_summary["name"] == nm)
        if not mask.any():
            continue
        n_is = len(grp)
        persistent = grp[grp["oos_sig"] == "significant (BH adjusted)"]
        n_oos = len(persistent)
        same_sign = (np.sign(grp["is_r"]) == np.sign(grp["oos_r"])).sum()
        idx_summary.loc[mask, "wf_is_sig"] = n_is
        idx_summary.loc[mask, "wf_oos_persistent"] = n_oos
        idx_summary.loc[mask, "wf_oos_pct"] = round(100 * n_oos / max(n_is, 1), 1)
        idx_summary.loc[mask, "wf_same_sign"] = same_sign
        idx_summary.loc[mask, "wf_sign_rate"] = round(100 * same_sign / max(n_is, 1), 1)
        if not persistent.empty:
            best_idx = persistent["oos_r"].abs().idxmax()
            idx_summary.loc[mask, "wf_best_oos_r"] = round(persistent.loc[best_idx, "oos_r"], 4)
            idx_summary.loc[mask, "wf_best_oos_p"] = persistent.loc[best_idx, "oos_p"]
    return idx_summary


all_idx_summaries = {}  # {ticker: merged idx_summary}
for ticker in TICKERS:
    per_filter_summaries = []
    for filt in CORE_FILTERS:
        label = filt["label"]
        key = (ticker, label)
        if key not in all_corr_results:
            continue
        print(f"\n--- {ticker} / {label} ---")
        idx_sum = build_index_summary(all_corr_results[key], all_coverage[key], ticker)
        idx_sum["filter_label"] = label
        n_sig = (idx_sum["score"] > 0).sum()
        print(f"  Index summary: {len(idx_sum)} indices, {n_sig} significant")

        print(f"  Running walk-forward validation...")
        wf_df = walk_forward_validate(all_sentiment[key], all_prices[ticker], ticker)
        if not wf_df.empty:
            wf_df["filter_label"] = label
        n_is = len(wf_df)
        n_oos = (wf_df["oos_sig"] == "significant (BH adjusted)").sum() if not wf_df.empty else 0
        print(f"  WF: {n_is} IS-sig matched, {n_oos} persistent OOS ({100 * n_oos / max(n_is, 1):.1f}%)")

        idx_sum = enrich_with_walkforward(idx_sum, wf_df)
        per_filter_summaries.append(idx_sum)

    if not per_filter_summaries:
        continue

    # Merge: pick best-scoring filter per (topic_level, name, index_type)
    all_rows = pd.concat(per_filter_summaries, ignore_index=True)
    key_cols = ["topic_level", "name", "index_type"]
    best_idx = all_rows.groupby(key_cols)["score"].idxmax()
    merged = all_rows.loc[best_idx].copy()
    merged.sort_values("score", ascending=False, inplace=True)
    merged.drop(columns=["rank"], errors="ignore", inplace=True)
    merged.insert(0, "rank", range(1, len(merged) + 1))
    all_idx_summaries[ticker] = merged

    fl_counts = merged["filter_label"].value_counts()
    print(f"\n  {ticker} merged: {len(merged)} indices -- filter wins: {fl_counts.to_dict()}")

print(f"\nTop 5 drivers for {sample_ticker}:")
all_idx_summaries[sample_ticker][
    ["rank", "name", "index_type", "score", "best_metric", "filter_label", "BOTH_best_r", "wf_oos_persistent"]
].head(5)

## Step 8: Cross-Ticker Assessment

We now aggregate results across all tickers into three summary tables:

1. **Overall Ranking** — composite grade (A–F) based on BH significance, correlation strength, and OOS persistence
2. **Top Sentiment Drivers Per Ticker** — the single highest-scoring topic for each asset
3. **Top Sentiment Group Drivers Per Ticker** — group-level aggregation showing breadth of signal

In [ ]:
def load_correlation_summary_from_idx(ticker: str, idx: pd.DataFrame, results_df: pd.DataFrame) -> dict:
    n_indices = len(idx)
    n_sig = (idx["score"] > 0).sum()
    n_sparse_sig = ((idx["sparse"]) & (idx["score"] > 0)).sum()

    best_r_both = idx["BOTH_best_r"].abs().max() if "BOTH_best_r" in idx else np.nan
    best_r_up = idx["UP_best_r"].abs().max() if "UP_best_r" in idx else np.nan
    best_r_down = idx["DOWN_best_r"].abs().max() if "DOWN_best_r" in idx else np.nan
    best_r_any = np.nanmax([best_r_both, best_r_up, best_r_down])

    top_score = idx["score"].max()
    top_row = idx.loc[idx["score"].idxmax()] if top_score > 0 else None
    top_driver = top_row["name"] if top_row is not None else "--"
    top_idx_type = top_row["index_type"] if top_row is not None else "--"
    top_best_metric = top_row["best_metric"] if top_row is not None and "best_metric" in idx else "--"
    top_filter = top_row["filter_label"] if top_row is not None and "filter_label" in idx else "--"

    sig_idx = idx[idx["score"] > 0]
    metric_wins = sig_idx["best_metric"].value_counts() if "best_metric" in sig_idx else pd.Series(dtype=int)
    dominant_metric = metric_wins.index[0] if len(metric_wins) > 0 else "--"

    filter_wins = sig_idx["filter_label"].value_counts() if "filter_label" in sig_idx else pd.Series(dtype=int)
    dominant_filter = filter_wins.index[0] if len(filter_wins) > 0 else "--"
    n_baseline_wins = int(filter_wins.get("baseline", 0))
    n_filtered_wins = int(n_sig - n_baseline_wins)

    total_tests = len(results_df)
    n_bh = (results_df["significance"] == "significant (BH adjusted)").sum()

    wf_is = int(idx["wf_is_sig"].sum()) if "wf_is_sig" in idx else 0
    wf_oos = int(idx["wf_oos_persistent"].sum()) if "wf_oos_persistent" in idx else 0
    wf_pct = round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0
    wf_sign = int(idx["wf_same_sign"].sum()) if "wf_same_sign" in idx else 0
    wf_sign_pct = round(100 * wf_sign / max(wf_is, 1), 1) if wf_is > 0 else 0.0

    best_oos_r = np.nan
    if "wf_best_oos_r" in idx:
        valid = idx["wf_best_oos_r"].dropna()
        if not valid.empty:
            best_oos_r = valid.abs().max()

    sig_regimes = idx[idx["score"] > 0]["sig_regimes"].value_counts()
    n_all_regime = sum(1 for v in sig_regimes.index if "BOTH" in v and "UP" in v and "DOWN" in v)
    n_regime_specific = n_sig - n_all_regime

    return {
        "ticker": ticker,
        "n_indices": n_indices,
        "total_corr_tests": total_tests,
        "n_bh_sig": n_bh,
        "bh_sig_pct": round(100 * n_bh / max(total_tests, 1), 1),
        "n_sig_indices": n_sig,
        "n_sparse_sig": n_sparse_sig,
        "top_score": top_score,
        "top_driver": top_driver,
        "top_idx_type": top_idx_type,
        "top_best_metric": top_best_metric,
        "top_filter": top_filter,
        "dominant_metric": dominant_metric,
        "dominant_filter": dominant_filter,
        "n_baseline_wins": n_baseline_wins,
        "n_filtered_wins": n_filtered_wins,
        "best_r_any": round(best_r_any, 4) if not np.isnan(best_r_any) else np.nan,
        "best_r_both": round(best_r_both, 4) if not np.isnan(best_r_both) else np.nan,
        "wf_is_sig": wf_is,
        "wf_oos_persistent": wf_oos,
        "wf_oos_pct": wf_pct,
        "wf_sign_consistency": wf_sign_pct,
        "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
        "n_all_regime_sig": n_all_regime,
        "n_regime_specific": n_regime_specific,
    }


def compute_overall_grade(row: dict) -> tuple[str, float]:
    score = 0.0
    score += min(row.get("bh_sig_pct", 0) / 30.0, 1.0) * 10
    score += min(row.get("n_sig_indices", 0) / 80.0, 1.0) * 10
    score += min(row.get("top_score", 0) / 50.0, 1.0) * 15
    best_r = row.get("best_r_any", 0)
    if np.isnan(best_r):
        best_r = 0
    score += min(best_r / 0.5, 1.0) * 10
    score += min(row.get("wf_oos_pct", 0) / 50.0, 1.0) * 30
    score += min(row.get("wf_sign_consistency", 0) / 80.0, 1.0) * 10
    oos_r = row.get("best_oos_r", 0)
    if np.isnan(oos_r):
        oos_r = 0
    score += min(oos_r / 0.5, 1.0) * 15
    grade = "A" if score >= 80 else "B" if score >= 65 else "C" if score >= 45 else "D" if score >= 25 else "F"
    return grade, round(score, 1)


def load_group_drivers(ticker: str, idx: pd.DataFrame) -> pd.DataFrame:
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        return pd.DataFrame()
    sig["group"] = sig["name"].str.split("-").str[0].str.strip()
    rows = []
    for group, grp in sig.groupby("group"):
        best_r_cols = [c for c in ["BOTH_best_r", "UP_best_r", "DOWN_best_r"] if c in grp]
        best_r = grp[best_r_cols].abs().max().max() if best_r_cols else np.nan
        best_score_idx = grp["score"].idxmax()
        best_name = grp.loc[best_score_idx, "name"]
        best_idx_type = grp.loc[best_score_idx, "index_type"]
        best_filter = grp.loc[best_score_idx, "filter_label"] if "filter_label" in grp else "--"
        wf_oos = int(grp["wf_oos_persistent"].sum()) if "wf_oos_persistent" in grp else 0
        wf_is = int(grp["wf_is_sig"].sum()) if "wf_is_sig" in grp else 0
        wf_pct = round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0
        best_oos_r = np.nan
        if "wf_best_oos_r" in grp:
            valid = grp["wf_best_oos_r"].dropna()
            if not valid.empty:
                best_oos_r = valid.abs().max()
        rows.append(
            {
                "ticker": ticker,
                "label": TICKER_LABELS.get(ticker, ticker),
                "group": group,
                "n_sig": len(grp),
                "top_score": grp["score"].max(),
                "best_r": round(best_r, 4) if not np.isnan(best_r) else np.nan,
                "best_driver": best_name,
                "best_idx_type": best_idx_type,
                "best_filter": best_filter,
                "wf_is_sig": wf_is,
                "wf_oos_persistent": wf_oos,
                "wf_oos_pct": wf_pct,
                "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("top_score", ascending=False)


ct_rows = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    # Combine all filter results for this ticker for total_tests/n_bh counting
    ticker_results = pd.concat(
        [v for (t, l), v in all_corr_results.items() if t == ticker],
        ignore_index=True,
    )
    row = load_correlation_summary_from_idx(ticker, all_idx_summaries[ticker], ticker_results)
    row["label"] = TICKER_LABELS.get(ticker, ticker)
    grade, score = compute_overall_grade(row)
    row["grade"] = grade
    row["overall_score"] = score
    ct_rows.append(row)

ct_df = pd.DataFrame(ct_rows)
ct_df.sort_values("overall_score", ascending=False, inplace=True)
ct_df.reset_index(drop=True, inplace=True)
ct_df.insert(0, "rank", range(1, len(ct_df) + 1))

group_frames = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    gdf = load_group_drivers(ticker, all_idx_summaries[ticker])
    if not gdf.empty:
        group_frames.append(gdf)
all_groups = pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()

print("Cross-ticker assessment built successfully.")
print(f"Tickers: {len(ct_df)}, Group driver rows: {len(all_groups)}")

### Table 1: Overall Ranking

In [ ]:
display_cols_1 = [
    "rank",
    "ticker",
    "label",
    "grade",
    "overall_score",
    "n_sig_indices",
    "bh_sig_pct",
    "best_r_any",
    "wf_oos_pct",
    "wf_sign_consistency",
    "best_oos_r",
    "dominant_metric",
    "dominant_filter",
]
ct_df[display_cols_1].rename(
    columns={
        "rank": "#",
        "label": "Asset",
        "grade": "Grade",
        "overall_score": "Score",
        "n_sig_indices": "Sig Idx",
        "bh_sig_pct": "BH %",
        "best_r_any": "Best |r|",
        "wf_oos_pct": "OOS %",
        "wf_sign_consistency": "Sign %",
        "best_oos_r": "OOS |r|",
    }
)

### Table 2: Top Sentiment Drivers Per Ticker

In [ ]:
display_cols_2 = [
    "rank",
    "ticker",
    "top_score",
    "top_driver",
    "top_idx_type",
    "top_best_metric",
    "top_filter",
    "best_r_both",
]
ct_df[display_cols_2].rename(
    columns={
        "rank": "#",
        "top_score": "Top Score",
        "top_driver": "Top Correlation Driver",
        "top_idx_type": "Index Type",
        "best_r_both": "|r| All",
    }
)

### Table 3: Top Sentiment Group Drivers Per Ticker

In [ ]:
if not all_groups.empty:
    display_cols_3 = [
        "ticker",
        "label",
        "group",
        "n_sig",
        "top_score",
        "best_r",
        "best_filter",
        "best_driver",
        "wf_oos_persistent",
        "wf_oos_pct",
        "best_oos_r",
    ]
    ipy_display(
        all_groups[display_cols_3].rename(
            columns={
                "label": "Asset",
                "n_sig": "Sig",
                "top_score": "Score",
                "best_r": "Best |r|",
                "best_driver": "Best Driver",
                "wf_oos_persistent": "OOS Sig",
                "wf_oos_pct": "OOS %",
                "best_oos_r": "OOS |r|",
            }
        )
    )
else:
    print("No group drivers found.")

### Regime Insights

Some sentiment drivers are significant across all market conditions, while others only work in **UP** (bullish) or **DOWN** (bearish) regimes. This distinction matters for portfolio construction:

- **Regime-universal drivers** are the most robust — they work regardless of trend direction and are safer to deploy in a live strategy.
- **Regime-specific drivers** (UP-only or DOWN-only) still carry signal, but require the strategy to know which regime it’s in. They can add value in a regime-conditional model but are riskier as standalone signals.

**Sign-flipping drivers** are a special case where the correlation *reverses direction* between UP and DOWN regimes. For example, positive supply sentiment might predict higher prices in a bull market (supply absorbed by strong demand) but lower prices in a bear market (oversupply into weakening demand). The same headline content has opposite implications depending on the trend.

For investors, sign-flippers are a double-edged sword:
- They carry **more information** than regime-neutral drivers because they’re predictive in *both* directions — but only if the model conditions on regime
- A naive strategy trained in a bull market will be **actively harmed** (not just noisy) by a sign-flipping driver in a bear market
- They are most common in **supply-side and macro topics**, where the same fundamental fact has opposite implications depending on economic conditions

In [ ]:
regime_rows = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    idx = all_idx_summaries[ticker]
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        continue

    up_only = sig[sig["sig_regimes"].str.contains("UP", na=False) & ~sig["sig_regimes"].str.contains("DOWN", na=False)]
    down_only = sig[
        sig["sig_regimes"].str.contains("DOWN", na=False) & ~sig["sig_regimes"].str.contains("UP", na=False)
    ]
    both_ud = sig[sig["sig_regimes"].str.contains("UP", na=False) & sig["sig_regimes"].str.contains("DOWN", na=False)]

    n_sign_flip = 0
    flip_examples = []
    for _, row in both_ud.iterrows():
        up_r = row.get("UP_best_r")
        down_r = row.get("DOWN_best_r")
        if pd.notna(up_r) and pd.notna(down_r) and up_r * down_r < 0:
            n_sign_flip += 1
            if len(flip_examples) < 2:
                flip_examples.append(f"{row['name']} (UP r={up_r:+.3f}, DOWN r={down_r:+.3f})")

    regime_rows.append(
        {
            "ticker": ticker,
            "label": TICKER_LABELS.get(ticker, ticker),
            "total_sig": len(sig),
            "UP only": len(up_only),
            "DOWN only": len(down_only),
            "both UP & DOWN": len(both_ud),
            "sign flips": n_sign_flip,
            "flip_examples": "; ".join(flip_examples) if flip_examples else "--",
        }
    )

regime_df = pd.DataFrame(regime_rows)

print("Regime Breakdown: How many significant drivers are regime-specific vs universal?\n")
ipy_display(
    regime_df[["ticker", "label", "total_sig", "UP only", "DOWN only", "both UP & DOWN", "sign flips"]].rename(
        columns={"label": "Asset", "total_sig": "Total Sig"}
    )
)

flips = regime_df[regime_df["sign flips"] > 0]
if not flips.empty:
    print("\nSign-Flipping Drivers (correlation reverses between UP and DOWN regimes):\n")
    for _, row in flips.iterrows():
        print(f"  {row['ticker']} ({row['label']}): {row['sign flips']} driver(s)")
        if row["flip_examples"] != "--":
            for ex in row["flip_examples"].split("; "):
                print(f"    \u2192 {ex}")
else:
    print("\nNo sign-flipping drivers detected across any ticker.")